In [1]:
import onnxruntime as ort
import numpy as np
from at2v.tokenizer import TagBPETokenizer

In [2]:
# config is not required anymore since the computation graph is known
# CONFIG_PATH = "../checkpoints/config_63fc21b89723d1ce_b0d065e705028cb3.json"
TOKENIZER_PATH = "../checkpoints/token_dataset_b0d065e705028cb3_vocab_size_5000_freq_3.json"
MODEL_PATH = "../checkpoints/anitag2vec_63fc21b89723d1ce_b0d065e705028cb3_i128_e30_s157043_b256_p1871744.onnx"

tagtok = TagBPETokenizer.load_from_file(TOKENIZER_PATH)
session = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"Input name: {input_name}, Output name: {output_name}")

Input name: x, Output name: y


In [3]:
# Since the model is tiny, inference is almost instant on CPU!
# e.g. 1000 examples is about 4 to 5s
max_len_cut = 128
examples = 1
dummy_input = np.random.randn(examples, max_len_cut).astype(np.int64)
outputs = session.run(["y"], {input_name: dummy_input})
output = np.array(outputs)[-1] # onnxruntime adds another dim for output's' (1, ...)
output.shape

(1, 128)

In [4]:
from typing import List

def tokenize(tags: List[str]):
    encs = [tagtok.encode_ids(tag) for tag in tags]
    inp = []
    sep_id = tagtok.sep_token_id()
    for i, enc in enumerate(encs):
        inp.extend(enc)
        if i < len(encs) - 1:
            inp.append(sep_id)
    if len(inp) > max_len_cut:
        inp = inp[:max_len_cut]
    pad_id = tagtok.pad_token_id()
    inp = inp + [pad_id] * (max_len_cut - len(inp))
    return np.array(inp, dtype=np.int64)

def run_inference(tags: List[List[str]]):
    inp = np.array([tokenize(curr) for curr in tags])
    # print(inp)
    outputs = session.run(["y"], {"x": inp})
    return np.array(outputs)[-1]

def parse(hastags: str):
    return [word[1:] for word in hastags.split() if word.startswith("#")]

def compare(a: str, b: str):
    ao = run_inference([parse(a)])
    bo = run_inference([parse(b)])
    ao = ao / np.linalg.norm(ao, axis=-1)
    bo = bo / np.linalg.norm(bo, axis=-1)
    howmuch = ao @ bo.T
    print(f"{howmuch.item():.2}: '{a}' vs '{b}'")

print("We lose about 1e-5 precision (see torch vs onnx below), ranking should be mostly fine")
compare("#nikke", "#blue_archive")
compare("#hayase_yuuka", "#blue_archive")
compare("#♀", "#girl")
compare("#♂", "#boy")
compare("#girl", "#boy")
compare("#♂", "#♀")
compare("#♂", "#unrelated")

We lose about 1e-5 precision (see torch vs onnx below), ranking should be mostly fine
0.21: '#nikke' vs '#blue_archive'
0.41: '#hayase_yuuka' vs '#blue_archive'
0.28: '#♀' vs '#girl'
0.12: '#♂' vs '#boy'
0.72: '#girl' vs '#boy'
0.66: '#♂' vs '#♀'
0.24: '#♂' vs '#unrelated'


In [5]:
import torch
from at2v.anitag2vec import AniTag2Vec, ModelConfig, AniTag2VecRunner
CONFIG_PATH = "../checkpoints/config_63fc21b89723d1ce_b0d065e705028cb3.json"
MODEL_PATH_TORCH = "../checkpoints/anitag2vec_63fc21b89723d1ce_b0d065e705028cb3_i128_e30_s157043_b256_p1871744.pth"
cfg = ModelConfig.load_from_file(CONFIG_PATH)
anitag2vec = AniTag2Vec(
    vocab_size=cfg.HYPERP_TAGTOK_VOCAB_SIZE,
    max_len_cut=cfg.HYPERP_TAGTOK_MAX_TOKEN_CLAMP,
    d_model=cfg.HYPERP_TRANSFORMER_D_MODEL,
    n_heads=cfg.HYPERP_TRANSFORMER_N_HEADS,
    n_layers=cfg.HYPERP_TRANSFORMER_N_LAYERS,
    output_emb=cfg.HYPERP_OUTPUT_EMB,
)
anitag2vec.load_state_dict(torch.load(MODEL_PATH_TORCH))
anitag2vec.eval()
runner = AniTag2VecRunner(tagtok, anitag2vec)

print("Comparison: ONNX export vs PyTorch model")
# ONNX
test = np.expand_dims(np.arange(128), axis=0)
onnx_out = np.array(session.run(["y"], {"x": test}))[-1]

# PyTorch
with torch.inference_mode():
    torch_out = anitag2vec(torch.from_numpy(test))
    torch_out = torch_out.detach().numpy()
print("ONNX", onnx_out[:, :10])
print("PyTorch", torch_out[:, :10])

def precision(onnx_out, torch_out, eps: float):
    print("All Eq at eps=", eps, np.all(np.abs(onnx_out - torch_out) < eps))
precision(onnx_out, torch_out, 1e-3)
precision(onnx_out, torch_out, 1e-4)
precision(onnx_out, torch_out, 1e-5)
precision(onnx_out, torch_out, 1e-6)

Comparison: ONNX export vs PyTorch model
ONNX [[-2.19328     0.34842864 -3.9214659   2.9389727  -4.018423   -6.566759
  -2.356137   -0.86571544  1.0988523  -1.8314098 ]]
PyTorch [[-2.1932783   0.34842816 -3.9214654   2.9389746  -4.0184245  -6.56676
  -2.3561378  -0.8657137   1.0988503  -1.8314109 ]]
All Eq at eps= 0.001 True
All Eq at eps= 0.0001 True
All Eq at eps= 1e-05 True
All Eq at eps= 1e-06 False


In [7]:
inp = ["#♀", "#♂", "#♂ #♀"]
torch_out = runner.run_inference_human(inp).detach().numpy()
onnx_out = run_inference([parse(x) for x in inp])

# print(torch_out[:, :7])
# print(onnx_out[:, :7])
precision(onnx_out, torch_out, 1e-1)
precision(onnx_out, torch_out, 1e-2)
precision(onnx_out, torch_out, 1e-4)
precision(onnx_out, torch_out, 1e-5)
precision(onnx_out, torch_out, 1e-6)
cp = (torch_out @ onnx_out.T) / (np.linalg.norm(torch_out, axis=-1) * np.linalg.norm(onnx_out, axis=-1))
np.diag(cp)

All Eq at eps= 0.1 True
All Eq at eps= 0.01 True
All Eq at eps= 0.0001 True
All Eq at eps= 1e-05 True
All Eq at eps= 1e-06 False


array([0.99999994, 1.        , 1.0000001 ], dtype=float32)